<a href="https://colab.research.google.com/github/gautamkushwaha/ML-and-Deep-Learning-Projects/blob/main/Image_captioning_BLIP_Processor.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models

class ImageCaptioner(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super(ImageCaptioner, self).__init__()

        # 1. ENCODER: Pre-trained ResNet
        resnet = models.resnet50(pretrained=True)
        # Remove the last classification layer (FC)
        modules = list(resnet.children())[:-1]
        self.resnet = nn.Sequential(*modules)
        # Map image features to hidden_size
        self.img_to_h = nn.Linear(resnet.fc.in_features, hidden_size)

        # 2. DECODER: Embedding + LSTM
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.lstm = nn.LSTM(embed_size, hidden_size, batch_first=True)
        self.fc_out = nn.Linear(hidden_size, vocab_size)

    def forward(self, images, captions):
        # Extract features: [Batch, 2048, 1, 1] -> [Batch, 2048]
        with torch.no_grad():
            features = self.resnet(images).reshape(images.shape[0], -1)

        # Initial hidden state from image
        h0 = self.img_to_h(features).unsqueeze(0) # [1, Batch, Hidden]
        c0 = torch.zeros_like(h0)                 # Initial cell state

        # Word Embeddings
        embeddings = self.embedding(captions)     # [Batch, SeqLen, Embed]

        # LSTM Step
        outputs, _ = self.lstm(embeddings, (h0, c0))

        # Map to Vocabulary
        return self.fc_out(outputs) # [Batch, SeqLen, VocabSize]

In [ ]:
def generate_caption(model, image, vocab, max_length=20):
    model.eval()
    result_caption = []

    with torch.no_grad():
        # 1. Get image features from CNN
        features = model.resnet(image).reshape(1, -1)
        h, c = model.img_to_h(features).unsqueeze(0), torch.zeros(1, 1, hidden_size)

        # 2. Start with the <START> token
        word_idx = torch.tensor([vocab['<START>']])

        for _ in range(max_length):
            # Embed word and run through LSTM
            embeddings = model.embedding(word_idx).unsqueeze(0) # [1, 1, Embed]
            output, (h, c) = model.lstm(embeddings, (h, c))

            # Get highest probability word (Greedy Search)
            output = model.fc_out(output.squeeze(0))
            predicted_idx = output.argmax(1).item()

            # If model says <END>, stop
            if vocab.idx_to_word[predicted_idx] == '<END>':
                break

            result_caption.append(vocab.idx_to_word[predicted_idx])

            # Prepare next input word
            word_idx = torch.tensor([predicted_idx])

    return ' '.join(result_caption)

In [ ]:
import torch
import torch.nn as nn
import torchvision.models as models
import torchvision.transforms as transforms
from PIL import Image

# 1. VOCABULARY HELPER
class Vocabulary:
    def __init__(self):
        # In a real scenario, this is built from the dataset
        self.word_to_idx = {"<PAD>": 0, "<START>": 1, "<END>": 2, "<UNK>": 3,
                            "a": 4, "dog": 5, "is": 6, "running": 7, "in": 8, "grass": 9}
        self.idx_to_word = {v: k for k, v in self.word_to_idx.items()}

# 2. THE MODEL (CNN ENCODER + RNN DECODER)
class CNN_RNN_Captioner(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size):
        super(CNN_RNN_Captioner, self).__init__()

        # Encoder: ResNet50
        resnet = models.resnet50(weights=models.ResNet50_Weights.DEFAULT)
        for param in resnet.parameters():
            param.requires_grad = False  # Freeze weights

        self.resnet = nn.Sequential(*list(resnet.children())[:-1])
        self.img_to_h = nn.Linear(resnet.fc.in_features, hidden_size)

        # Decoder: LSTM
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.lstm = nn.LSTM(embed_size, hidden_size, batch_first=True)
        self.fc_out = nn.Linear(hidden_size, vocab_size)

    def forward(self, image, max_length, vocab):
        caption = []
        with torch.no_grad():
            # Extract image features
            features = self.resnet(image).reshape(1, -1)
            h, c = self.img_to_h(features).unsqueeze(0), torch.zeros(1, 1, 512)

            # Start token
            word_idx = torch.tensor([vocab.word_to_idx["<START>"]])

            for _ in range(max_length):
                # Generate next word
                embed = self.embedding(word_idx).unsqueeze(0)
                output, (h, c) = self.lstm(embed, (h, c))
                output = self.fc_out(output.squeeze(0))

                predicted = output.argmax(1).item()
                word = vocab.idx_to_word[predicted]

                if word == "<END>": break
                caption.append(word)
                word_idx = torch.tensor([predicted])

        return " ".join(caption)

# 3. EXECUTION LOGIC
def main(image_path):
    # Setup
    vocab = Vocabulary()
    model = CNN_RNN_Captioner(vocab_size=len(vocab.word_to_idx), embed_size=256, hidden_size=512)
    model.eval()

    # Image Pre-processing
    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
        transforms.Normalize((0.485, 0.456, 0.406), (0.229, 0.224, 0.225))
    ])

    img = Image.open(image_path).convert("RGB")
    img_tensor = transform(img).unsqueeze(0) # Add batch dimension

    # Generate!
    caption = model.forward(img_tensor, max_length=10, vocab=vocab)
    print(f"Generated Caption: {caption}")

# Note: This will produce random words until trained on a dataset like COCO.
main('./cat.jpeg')

Generated Caption: <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK> <UNK>


In [ ]:
!pip install transformers torch pillow

In [ ]:
import torch
from PIL import Image
from transformers import BlipProcessor, BlipForConditionalGeneration

# 1. Load the pretrained model and processor
# 'base' is smaller and faster; 'large' is more accurate
processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")

def get_real_caption(image_path):
    # 2. Load and prep the image
    raw_image = Image.open(image_path).convert('RGB')

    # 3. Process the image for the model
    inputs = processor(raw_image, return_tensors="pt")

    # 4. Generate the caption
    out = model.generate(**inputs)

    # 5. Decode the numbers back into a sentence
    caption = processor.decode(out[0], skip_special_tokens=True)
    return caption

# Run it on your cat image!
print(get_real_caption('your_image.jpg'))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


preprocessor_config.json:   0%|          | 0.00/287 [00:00<?, ?B/s]

The image processor of type `BlipImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/506 [00:00<?, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/990M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/473 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie text_decoder.cls.predictions.bias to text_decoder.cls.predictions.decoder.bias, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie text_decoder.bert.embeddings.word_embeddings.weight to text_decoder.cls.predictions.decoder.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
BlipForConditionalGeneration LOAD REPORT from: Salesforce/blip-image-captioning-base
Key                                       | Status     |  | 
------------------------------------------+------------+--+-
text_decoder.bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identic

FileNotFoundError: [Errno 2] No such file or directory: 'your_image.jpg'

In [ ]:
print(get_real_caption('./dog.png'))

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


a golden retrieve dog sitting down with its tongue out


In [ ]:
print(get_real_caption('./cat.jpeg'))

a cat laying down on the floor
